# Programación orientada a objetos con Python
## Video 6. Herencia: `super()`, sobrescritura y orden de resolución de métodos

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Modelar relaciones de especialización («es un caso particular de») mediante herencia entre clases matemáticas.
2. Reutilizar el inicializador y los métodos de la superclase mediante `super()`.
3. Sobrescribir métodos en una subclase para añadir fórmulas cerradas o comportamientos especializados.
4. Inspeccionar el orden de resolución de métodos (`Clase.mro()`) y entender cómo Python recorre una jerarquía de clases.

## 1. Planteamiento del problema

**Definición.** Un *polinomio cuadrático* $P(x) = a_0 + a_1 x + a_2 x^2$ con $a_2 \neq 0$ es un caso particular de polinomio de grado $2$. Su *discriminante* es

$$\Delta = a_1^2 - 4 a_2 a_0,$$

y la abscisa del *vértice* de su parábola es $x_v = -\frac{a_1}{2 a_2}$. Cuando $\Delta \ge 0$, sus raíces reales están dadas por la fórmula cuadrática cerrada

$$x = \frac{-a_1 \pm \sqrt{\Delta}}{2 a_2}.$$

De forma análoga, $\mathbb{R}^2$ y $\mathbb{R}^3$ son casos particulares del espacio vectorial general $\mathbb{R}^n$.

**Problema: especialización sin duplicar código.**

- **Entrada:** las clases generales `Polinomio`, `Vector` y `Fraccion`.
- **Salida:** subclases especializadas (`PolinomioLineal`, `PolinomioCuadratico`, `VectorPlano`, `VectorEspacio`, `EnteroRacional`) que hereden toda la aritmética general e incorporen operaciones propias de su grado, dimensión o dominio.

## 2. Superclase, subclase y delegación con `super()`

Cuando una clase `Subclase` hereda de `Superclase` mediante `class Subclase(Superclase):`, adquiere automáticamente todos los métodos y propiedades de la superclase.

Dentro de la subclase, la función `super()` devuelve un objeto delegado que permite invocar la implementación de la superclase sin escribir explícitamente su nombre. Su uso más frecuente está en `__init__`: la subclase valida sus restricciones específicas y luego delega la inicialización general a `super().__init__(...)`.

In [ ]:
from math import gcd, sqrt


class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self._coeficientes = tuple(lista)

    @property
    def coeficientes(self):
        return self._coeficientes

    @property
    def grado(self):
        return len(self._coeficientes) - 1

    def evaluar(self, valor):
        resultado = 0
        for coeficiente in reversed(self._coeficientes):
            resultado = resultado * valor + coeficiente
        return resultado

    def derivar(self):
        if self.grado == 0:
            return Polinomio([0])
        return Polinomio(
            exponente * coeficiente
            for exponente, coeficiente in enumerate(self._coeficientes)
            if exponente > 0
        )

    def describir(self):
        return f"Polinomio de grado {self.grado}: {self}"

    def __repr__(self):
        return f"{type(self).__name__}({list(self._coeficientes)})"

    def __str__(self):
        return " + ".join(
            f"{coeficiente}*{self.variable}^{exponente}"
            for exponente, coeficiente in enumerate(self._coeficientes)
        )

    def __eq__(self, otro):
        if not isinstance(otro, Polinomio):
            return NotImplemented
        return self._coeficientes == otro._coeficientes

    def __add__(self, otro):
        if not isinstance(otro, Polinomio):
            return NotImplemented
        limite = max(len(self._coeficientes), len(otro._coeficientes))
        suma = [
            (self._coeficientes[i] if i < len(self._coeficientes) else 0)
            + (otro._coeficientes[i] if i < len(otro._coeficientes) else 0)
            for i in range(limite)
        ]
        return Polinomio(suma)


class PolinomioLineal(Polinomio):
    def __init__(self, pendiente, ordenada):
        if pendiente == 0:
            raise ValueError("En un polinomio lineal la pendiente no puede ser cero.")
        super().__init__([ordenada, pendiente])

    @property
    def pendiente(self):
        return self._coeficientes[1]

    @property
    def ordenada(self):
        return self._coeficientes[0]


class PolinomioCuadratico(Polinomio):
    def __init__(self, a, b, c):
        if a == 0:
            raise ValueError("El coeficiente cuadrático no puede ser cero.")
        super().__init__([c, b, a])

    @property
    def discriminante(self):
        c, b, a = self._coeficientes
        return b ** 2 - 4 * a * c

    @property
    def vertice_x(self):
        _, b, a = self._coeficientes
        return -b / (2 * a)


recta = PolinomioLineal(2, -4)
parabola = PolinomioCuadratico(1, -4, 3)

print("Recta:", recta, "| evaluar en 3:", recta.evaluar(3))
print("Parábola:", parabola, "| derivada:", parabola.derivar())
print("Discriminante:", parabola.discriminante, "| abscisa del vértice:", parabola.vertice_x)
print("isinstance(parabola, Polinomio):", isinstance(parabola, Polinomio))
print("issubclass(PolinomioCuadratico, Polinomio):", issubclass(PolinomioCuadratico, Polinomio))

## 3. Sobrescritura de métodos y especialización

Una subclase puede **sobrescribir** un método heredado definiendo otro con el mismo nombre. En lugar de descartar por completo la versión de la superclase, es una práctica frecuente invocar `super().metodo(...)` para ampliar o enriquecer el resultado base.

In [ ]:
class PolinomioLineal(Polinomio):
    def __init__(self, pendiente, ordenada):
        if pendiente == 0:
            raise ValueError("En un polinomio lineal la pendiente no puede ser cero.")
        super().__init__([ordenada, pendiente])

    def raiz(self):
        b, m = self._coeficientes
        return -b / m

    def describir(self):
        base = super().describir()
        return f"{base} | raíz única x = {self.raiz()}"


class PolinomioCuadratico(Polinomio):
    def __init__(self, a, b, c):
        if a == 0:
            raise ValueError("El coeficiente cuadrático no puede ser cero.")
        super().__init__([c, b, a])

    @property
    def discriminante(self):
        c, b, a = self._coeficientes
        return b ** 2 - 4 * a * c

    @property
    def vertice_x(self):
        _, b, a = self._coeficientes
        return -b / (2 * a)

    def describir(self):
        base = super().describir()
        return f"{base} | discriminante = {self.discriminante}, vértice en x = {self.vertice_x}"


print(Polinomio([1, 0, 0, 2]).describir())
print(PolinomioLineal(2, -4).describir())
print(PolinomioCuadratico(1, -4, 3).describir())

## 4. Orden de resolución de métodos (`mro`) y clases mezcladoras

Cuando se busca un método o atributo sobre una instancia, Python recorre las clases de la jerarquía siguiendo una lista lineal determinista llamada **orden de resolución de métodos** (*Method Resolution Order*, accesible mediante `Clase.mro()`).

Esta propiedad es fundamental en la herencia múltiple, por ejemplo al combinar una clase matemática principal con una **clase mezcladora** (*mixin*) que aporta una capacidad transversal, como generar tablas de valores. Todas las jerarquías desembocan en la raíz común `object`.

In [ ]:
class EvaluacionTabularMixin:
    def tabla_de_valores(self, nodos):
        return [(x, self.evaluar(x)) for x in nodos]


class PolinomioCuadraticoTabular(PolinomioCuadratico, EvaluacionTabularMixin):
    pass


print("Orden de resolución de métodos:")
for clase in PolinomioCuadraticoTabular.mro():
    print(" ->", clase.__name__)

cuadratico = PolinomioCuadraticoTabular(1, -4, 3)
print("Tabla en [0, 1, 2, 3, 4]:", cuadratico.tabla_de_valores([0, 1, 2, 3, 4]))

### Tabla comparativa de mecanismos de herencia

| Elemento | Propósito | Cómo se escribe | Ejemplo en nuestra serie |
|---|---|---|---|
| Herencia simple | Especializar una clase base | `class Sub(Base):` | `PolinomioCuadratico(Polinomio)` |
| Delegación con `super()` | Reutilizar `__init__` o métodos de la base | `super().__init__(...)` | `super().__init__([c, b, a])` |
| Sobrescritura | Reemplazar o ampliar un método heredado | Redefinir el método en la subclase | `PolinomioCuadratico.describir` |
| Verificación de tipos | Consultar pertenencia a jerarquía | `isinstance(obj, Base)` / `issubclass(Sub, Base)` | `isinstance(parabola, Polinomio)` |
| Orden de resolución (`mro`) | Consultar la ruta de búsqueda de métodos | `Clase.mro()` | `PolinomioCuadraticoTabular.mro()` |

## 5. Ejercicios

**Ejercicio 1 (`VectorPlano` y `VectorEspacio` como subclases de `Vector`).** Parte de la clase general `Vector` en $\mathbb{R}^n$ con los métodos `__len__`, `__eq__`, `__add__`, `norma()` y `producto_interior(otro)`:
- **Entrada:** coordenadas $(x, y) \in \mathbb{R}^2$ para `VectorPlano(Vector)` y coordenadas $(x, y, z) \in \mathbb{R}^3$ para `VectorEspacio(Vector)`.
- **Salida:**
  - `VectorPlano` delega `super().__init__([x, y])`, expone las propiedades `x` e `y`, y define `rotar_90_grados(self)` devolviendo `VectorPlano(-self.y, self.x)`.
  - `VectorEspacio` delega `super().__init__([x, y, z])`, expone `x`, `y`, `z`, y define `producto_vectorial(self, otro)`.

**Ejercicio 2 (cálculo exacto de raíces reales en `PolinomioLineal` y `PolinomioCuadratico`).** Añade el método `raices_reales(self)` a ambas subclases:
- **Entrada:** una instancia de `PolinomioLineal` o `PolinomioCuadratico`.
- **Salida:** una tupla ordenada con las raíces reales distintas:
  - En `PolinomioLineal`: una tupla de un elemento `(raiz,)`.
  - En `PolinomioCuadratico`: tupla vacía `()` si $\Delta < 0$, tupla de un elemento `(vertice_x,)` si $\Delta = 0$, y tupla de dos elementos `(r1, r2)` con $r_1 < r_2$ si $\Delta > 0$.

**Ejercicio 3 (`EnteroRacional(Fraccion)` y análisis de `mro()`).** Partiendo de una clase `Fraccion` con `__init__(self, numerador, denominador=1)`, `__repr__`, `__str__`, `__eq__` y `__add__`, define la subclase `EnteroRacional(Fraccion)`:
- **Entrada:** un único número entero `valor` en `EnteroRacional(valor)`.
- **Salida:** `EnteroRacional` llama a `super().__init__(valor, 1)`, sobrescribe `__repr__` para devolver `f"EnteroRacional({self.numerador})"`, añade `es_par(self)`, y permite operar con `Fraccion` mediante `+` y `==`. Muestra `EnteroRacional.mro()`.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class Vector:
    def __init__(self, componentes):
        lista = [float(valor) for valor in componentes]
        if not lista:
            raise ValueError("Un vector debe tener al menos una componente.")
        self._componentes = tuple(lista)

    @property
    def componentes(self):
        return self._componentes

    def __len__(self):
        return len(self._componentes)

    def __repr__(self):
        return f"{type(self).__name__}({list(self._componentes)})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector):
            return NotImplemented
        return self._componentes == otro._componentes

    def __add__(self, otro):
        if not isinstance(otro, Vector):
            return NotImplemented
        if len(self) != len(otro):
            raise ValueError("Dimensiones incompatibles para la suma.")
        return Vector(a + b for a, b in zip(self._componentes, otro._componentes))

    def norma(self):
        return sqrt(sum(c ** 2 for c in self._componentes))

    def producto_interior(self, otro):
        if len(self) != len(otro):
            raise ValueError("Dimensiones incompatibles para el producto interior.")
        return sum(a * b for a, b in zip(self._componentes, otro._componentes))


class VectorPlano(Vector):
    def __init__(self, x, y):
        super().__init__([x, y])

    @property
    def x(self):
        return self._componentes[0]

    @property
    def y(self):
        return self._componentes[1]

    def rotar_90_grados(self):
        return VectorPlano(-self.y, self.x)


class VectorEspacio(Vector):
    def __init__(self, x, y, z):
        super().__init__([x, y, z])

    @property
    def x(self):
        return self._componentes[0]

    @property
    def y(self):
        return self._componentes[1]

    @property
    def z(self):
        return self._componentes[2]

    def producto_vectorial(self, otro):
        if not isinstance(otro, VectorEspacio):
            raise TypeError("Se esperaba otra instancia de VectorEspacio.")
        return VectorEspacio(
            self.y * otro.z - self.z * otro.y,
            self.z * otro.x - self.x * otro.z,
            self.x * otro.y - self.y * otro.x,
        )


u2 = VectorPlano(3, 4)
u2_rot = u2.rotar_90_grados()
print("VectorPlano:", u2, "| rotado 90 grados:", u2_rot, "| producto interior:", u2.producto_interior(u2_rot))

i = VectorEspacio(1, 0, 0)
j = VectorEspacio(0, 1, 0)
print("VectorEspacio i x j:", i.producto_vectorial(j), "| norma:", i.producto_vectorial(j).norma())

In [ ]:
# Ejercicio 2
class PolinomioLinealConRaices(PolinomioLineal):
    def raices_reales(self):
        return (self.raiz(),)


class PolinomioCuadraticoConRaices(PolinomioCuadratico):
    def raices_reales(self):
        _, b, a = self._coeficientes
        delta = self.discriminante
        if delta < 0:
            return ()
        if delta == 0:
            return (self.vertice_x,)
        raiz_delta = sqrt(delta)
        r1 = (-b - raiz_delta) / (2 * a)
        r2 = (-b + raiz_delta) / (2 * a)
        return tuple(sorted((r1, r2)))


print("Raíces de 2x - 6:", PolinomioLinealConRaices(2, -6).raices_reales())
print("Raíces de x^2 - 4x + 3:", PolinomioCuadraticoConRaices(1, -4, 3).raices_reales())
print("Raíces de x^2 - 2x + 1:", PolinomioCuadraticoConRaices(1, -2, 1).raices_reales())
print("Raíces de x^2 + 1:", PolinomioCuadraticoConRaices(1, 0, 1).raices_reales())

In [ ]:
# Ejercicio 3
class FraccionBase:
    def __init__(self, numerador, denominador=1):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def __repr__(self):
        return f"FraccionBase({self._numerador}, {self._denominador})"

    def __str__(self):
        return str(self._numerador) if self._denominador == 1 else f"{self._numerador}/{self._denominador}"

    def __eq__(self, otra):
        if not isinstance(otra, FraccionBase):
            return NotImplemented
        return self._numerador == otra._numerador and self._denominador == otra._denominador

    def __add__(self, otra):
        if not isinstance(otra, FraccionBase):
            return NotImplemented
        return FraccionBase(
            self._numerador * otra._denominador + self._denominador * otra._numerador,
            self._denominador * otra._denominador,
        )


class EnteroRacional(FraccionBase):
    def __init__(self, valor):
        super().__init__(valor, 1)

    def __repr__(self):
        return f"EnteroRacional({self.numerador})"

    def es_par(self):
        return self.numerador % 2 == 0


cuatro = EnteroRacional(4)
tercio = FraccionBase(1, 3)
print("Representación:", repr(cuatro), "| str:", str(cuatro), "| ¿es par?", cuatro.es_par())
print("4 + 1/3 =", cuatro + tercio)
print("¿EnteroRacional(4) == FraccionBase(8, 2)?", cuatro == FraccionBase(8, 2))
print("MRO de EnteroRacional:", [clase.__name__ for clase in EnteroRacional.mro()])

## 7. Resumen

- La **herencia** permite definir subclases especializadas reutilizando todos los métodos y propiedades de una superclase.
- `super()` delega en la siguiente clase del orden de resolución de métodos sin acoplar el código al nombre explícito de la superclase.
- La **sobrescritura** adapta métodos generales (como `describir`) a casos particulares sin perder la lógica común.
- `Clase.mro()` muestra la secuencia lineal exacta que sigue Python para resolver atributos y llamadas `super()`.
- En el siguiente video formalizaremos contratos compartidos entre clases distintas mediante polimorfismo y clases abstractas (`abc`).